# In Class Activity 4/9
## Dirks Wright

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
creditcard = pd.read_csv('creditcard.csv')
creditcard.head()


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


In [5]:
report = sv.analyze(creditcard)
report.show_html('credit_card.html')

                                             |          | [  0%]   00:00 -> (? left)

Report credit_card.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


The strongest associations appear to be within the 6 different pay variables and within the 6 different bill amount variables. It is important to note that there are almost no strong associations between a pay variable and a bill amount variable. 

Additionally, instances of categorical variables have been labeled with numbers instead of an informative value. For example sex has values 1 and 2, however, without a data dictionary it is almost impossible to determine the meaning of these.

Majority of the 6 different bill amounts exhibit a rightly skewed distribution

In [3]:
# baseline models
# Prepare the data
X = creditcard.drop('default payment next month', axis=1)
y = creditcard['default payment next month']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=222)

# Initialize models with default parameters
rf = RandomForestClassifier(random_state=222)
xgb = XGBClassifier(random_state=222)

# Train and evaluate on entire dataset
print("Random Forest on entire dataset:")
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(classification_report(y_test, rf_pred))

print("\nXGBoost on entire dataset:")
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
print(classification_report(y_test, xgb_pred))

# 5-fold cross-validation
from sklearn.model_selection import cross_val_score

rf_cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
xgb_cv_scores = cross_val_score(xgb, X, y, cv=5, scoring='accuracy')

print(f"\nRandom Forest CV - Mean accuracy: {rf_cv_scores.mean():.4f}, Std: {rf_cv_scores.std():.4f}")
print(f"XGBoost CV - Mean accuracy: {xgb_cv_scores.mean():.4f}, Std: {xgb_cv_scores.std():.4f}")

Random Forest on entire dataset:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      3744
           1       0.66      0.35      0.45      1056

    accuracy                           0.82      4800
   macro avg       0.75      0.65      0.67      4800
weighted avg       0.80      0.82      0.79      4800


XGBoost on entire dataset:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3744
           1       0.63      0.35      0.45      1056

    accuracy                           0.81      4800
   macro avg       0.73      0.64      0.67      4800
weighted avg       0.79      0.81      0.79      4800


Random Forest CV - Mean accuracy: 0.8137, Std: 0.0028
XGBoost CV - Mean accuracy: 0.8128, Std: 0.0035


The mean accuracy across the 5 folds indicates the average performance of the model on unseen data, providing a more robust estimate than a single train-test split. The standard deviation shows the variability in performance across different folds - a lower standard deviation suggests more consistent performance, while a higher one indicates the model is sensitive to the specific data split. This helps assess model stability and generalizability.

In [5]:
# building models with cross validation
rf_cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
xgb_cv_scores = cross_val_score(xgb, X, y, cv=5, scoring='accuracy')
print(f"\nRandom Forest CV - Mean accuracy: {rf_cv_scores.mean():.4f}, Std: {rf_cv_scores.std():.4f}")
print(f"XGBoost CV - Mean accuracy: {xgb_cv_scores.mean():.4f}, Std: {xgb_cv_scores.std():.4f}")

#print cross validations scores for each fold
print(f"Random Forest CV scores for each fold: {rf_cv_scores}")
print(f"XGBoost CV scores for each fold: {xgb_cv_scores}")




Random Forest CV - Mean accuracy: 0.8137, Std: 0.0028
XGBoost CV - Mean accuracy: 0.8128, Std: 0.0035
Random Forest CV scores for each fold: [0.81375    0.81145833 0.81520833 0.81       0.81791667]
XGBoost CV scores for each fold: [0.80979167 0.81083333 0.814375   0.81020833 0.81895833]


Low standard deviation across the folds and almost identical mean accuracy scores indicate consistency in both models predictive ability.

After looking at the classification report, it shows both models are biased at predicting 0 over 1. Given the circumstances of this case, inefficiently predicting 1 (instance of credit card fraud/no payment) it would be more beneficial to overpredict 1's rather than 0's to ensure credit card companies catch all risky accounts.  

In [10]:
#repeat baseline modeling using recall as the metric
rf_cv_recall = cross_val_score(rf, X, y, cv=5, scoring='recall')
xgb_cv_recall = cross_val_score(xgb, X, y, cv=5, scoring='recall')
print(f"\nRandom Forest CV - Mean recall: {rf_cv_recall.mean():.4f}, Std: {rf_cv_recall.std():.4f}")
print(f"XGBoost CV - Mean recall: {xgb_cv_recall.mean():.4f}, Std: {xgb_cv_recall.std():.4f}")

#print cross validation recall scores for each fold
print(f"Random Forest CV recall scores for each fold: {rf_cv_recall}")
print(f"XGBoost CV recall scores for each fold: {xgb_cv_recall}")



Random Forest CV - Mean recall: 0.3639, Std: 0.0035
XGBoost CV - Mean recall: 0.3686, Std: 0.0065
Random Forest CV recall scores for each fold: [0.35815269 0.36158192 0.36723164 0.36723164 0.3653484 ]
XGBoost CV recall scores for each fold: [0.35626767 0.37288136 0.37193974 0.36817326 0.37382298]


In [6]:
# rebuild rf and xgb models with adjustment to class weight to address class imbalance
rf_weighted = RandomForestClassifier(random_state=222, class_weight='balanced')
rf_weighted.fit(X_train, y_train)
rf_weighted_pred = rf_weighted.predict(X_test)
xgb_weighted = XGBClassifier(random_state=222, scale_pos_weight=10)
xgb_weighted.fit(X_train, y_train)
xgb_weighted_pred = xgb_weighted.predict(X_test)
print("\nRandom Forest with class weight adjustment:") 
print(classification_report(y_test, rf_weighted_pred))
print("\nXGBoost with class weight adjustment:")
print(classification_report(y_test, xgb_weighted_pred))



Random Forest with class weight adjustment:
              precision    recall  f1-score   support

           0       0.83      0.95      0.89      3744
           1       0.65      0.31      0.42      1056

    accuracy                           0.81      4800
   macro avg       0.74      0.63      0.65      4800
weighted avg       0.79      0.81      0.79      4800


XGBoost with class weight adjustment:
              precision    recall  f1-score   support

           0       0.89      0.62      0.73      3744
           1       0.35      0.74      0.48      1056

    accuracy                           0.65      4800
   macro avg       0.62      0.68      0.61      4800
weighted avg       0.78      0.65      0.68      4800



Adjusting for class imbalance improves XGBoost recall predictive ability more so than the Random Forest's predictive ability.


In [4]:
# XGBoost boostrapping (subsample =0.8)
xgb_bootstrap = XGBClassifier(random_state=222, scale_pos_weight=10, subsample=0.8)
xgb_bootstrap.fit(X_train, y_train)
xgb_bootstrap_pred = xgb_bootstrap.predict(X_test)
print("\nXGBoost with class weight adjustment and bootstrapping:")
print(classification_report(y_test, xgb_bootstrap_pred))



XGBoost with class weight adjustment and bootstrapping:
              precision    recall  f1-score   support

           0       0.89      0.63      0.74      3744
           1       0.36      0.73      0.48      1056

    accuracy                           0.65      4800
   macro avg       0.63      0.68      0.61      4800
weighted avg       0.78      0.65      0.68      4800



Bootstrapping had minnimal impact on the XGBoost model, with almost identical metrics. 

In [7]:
# hyperparameter tuning for Random Forest with 2 hyperparameters and 2 values each
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10]
}
grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=5, scoring='recall')
grid_rf.fit(X_train, y_train)   
print(f"Best parameters for Random Forest: {grid_rf.best_params_}")
best_rf = grid_rf.best_estimator_
best_rf_pred = best_rf.predict(X_test)
print("\nRandom Forest after hyperparameter tuning:")
print(classification_report(y_test, best_rf_pred))

# hyperparameter tuning for XGBoost with 2 hyperparameters and 2 values each
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6]
}
grid_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv=5, scoring='recall')
grid_xgb.fit(X_train, y_train)
print(f"Best parameters for XGBoost: {grid_xgb.best_params_}")
best_xgb = grid_xgb.best_estimator_
best_xgb_pred = best_xgb.predict(X_test)
print("\nXGBoost after hyperparameter tuning:")
print(classification_report(y_test, best_xgb_pred))


Best parameters for Random Forest: {'max_depth': None, 'n_estimators': 200}

Random Forest after hyperparameter tuning:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      3744
           1       0.65      0.35      0.46      1056

    accuracy                           0.82      4800
   macro avg       0.75      0.65      0.67      4800
weighted avg       0.80      0.82      0.79      4800

Best parameters for XGBoost: {'max_depth': 6, 'n_estimators': 200}

XGBoost after hyperparameter tuning:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3744
           1       0.60      0.36      0.45      1056

    accuracy                           0.81      4800
   macro avg       0.72      0.65      0.67      4800
weighted avg       0.79      0.81      0.79      4800



After briefly tuning hyperparameters for each model, specifically the number of estimators (n_estimators) and maximum depth (max_depth), the Random Forest classifier showed a slight improvement in the minority class F1-score while maintaining similar overall accuracy, indicating effective parameter choices. In contrast, XGBoost showed little to no improvement after tuning these parameters, with performance remaining largely unchanged. Additionally, applying class imbalance adjustments in XGBoost increased recall but significantly reduced overall accuracy, suggesting a poor trade-off. Overall, the selected parameters were appropriate for Random Forest, while XGBoost likely requires tuning of additional parameters to achieve meaningful improvemen

The Random Forest classifier performed slightly better than XGBoost out of the box, achieving comporable or slightly higher accuracy After hyperparameter tuning, Random Forest maintained or slightly improved its performance, while XGBoost showed minimal improvement. Overall, Random Forest consistently outperformed XGBoost across both baseline and tuned settings.